In [2]:
%pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 23.8 MB/s eta 0:00:0031m24.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 19.6 MB/s eta 0:00:0031m22.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]━ 3/4 [geopandas]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

In [7]:
plants = pd.read_excel(
    "~/Desktop/eia8602024/2___Plant_Y2024.xlsx",
    header=1
)

In [8]:
plants.head()

,Utility ID,Utility Name,Plant Code,Plant Name,Street Address,City,State,Zip,County,Latitude,...,Grid Voltage 2 (kV),Grid Voltage 3 (kV),Energy Storage,Natural Gas LDC Name,Natural Gas Pipeline Name 1,Natural Gas Pipeline Name 2,Natural Gas Pipeline Name 3,Pipeline Notes,Natural Gas Storage,Liquefied Natural Gas Storage
0,63560,"Sand Point Generating, LLC",1,Sand Point,100 Power Plant Way,Sand Point,AK,99661,Aleutians East,55.339722,...,,,N,NaN,NaN,NaN,NaN,NaN,N,X
1,195,Alabama Power Co,2,Bankhead Dam,19001 Lock 17 Road,Northport,AL,35476,Tuscaloosa,33.458665,...,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,195,Alabama Power Co,3,Barry,North Highway 43,Bucks,AL,36512,Mobile,31.0069,...,,,N,NaN,BAY GAS STORAGE,NaN,NaN,NaN,N,X
3,195,Alabama Power Co,4,Walter Bouldin Dam,750 Bouldin Dam Road,Wetumpka,AL,36092,Elmore,32.583889,...,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,195,Alabama Power Co,7,Gadsden,1000 Goodyear Ave,Gadsden,AL,35903,Etowah,34.0128,...,,,N,NaN,SOUTHERN NATURAL GAS COMPANY,NaN,NaN,NaN,N,X


In [9]:
plants.columns

Index(['Utility ID', 'Utility Name', 'Plant Code', 'Plant Name',
       'Street Address', 'City', 'State', 'Zip', 'County', 'Latitude',
       'Longitude', 'NERC Region', 'Balancing Authority Code',
       'Balancing Authority Name', 'Name of Water Source',
       'Primary Purpose (NAICS Code)', 'Regulatory Status', 'Sector',
       'Sector Name', 'FERC Cogeneration Status',
       'FERC Cogeneration Docket Number', 'FERC Small Power Producer Status',
       'FERC Small Power Producer Docket Number',
       'FERC Exempt Wholesale Generator Status',
       'FERC Exempt Wholesale Generator Docket Number', 'Ash Impoundment?',
       'Ash Impoundment Lined?', 'Ash Impoundment Status',
       'Transmission or Distribution System Owner',
       'Transmission or Distribution System Owner ID',
       'Transmission or Distribution System Owner State', 'Grid Voltage (kV)',
       'Grid Voltage 2 (kV)', 'Grid Voltage 3 (kV)', 'Energy Storage',
       'Natural Gas LDC Name', 'Natural Gas Pipeline 

In [10]:
plants_clean = plants[[
    "Plant Code",
    "Plant Name",
    "State",
    "County",
    "Latitude",
    "Longitude"
]]

In [13]:
plants_clean = plants_clean.dropna(subset=["Latitude","Longitude"])

In [15]:
print(plants_clean.head())
print(plants_clean.shape)

   Plant Code          Plant Name State          County   Latitude   Longitude
0           1          Sand Point    AK  Aleutians East  55.339722 -160.497222
1           2        Bankhead Dam    AL      Tuscaloosa  33.458665  -87.356823
2           3               Barry    AL          Mobile    31.0069    -88.0103
3           4  Walter Bouldin Dam    AL          Elmore  32.583889  -86.283056
4           7             Gadsden    AL          Etowah    34.0128    -85.9708
(16132, 6)


In [16]:
generators = pd.read_excel(
    "~/Desktop/eia8602024/3_1_Generator_Y2024.xlsx",
    header=1
)

In [17]:
generators.columns

Index(['Utility ID', 'Utility Name', 'Plant Code', 'Plant Name', 'State',
       'County', 'Generator ID', 'Technology', 'Prime Mover', 'Unit Code',
       'Ownership', 'Duct Burners',
       'Can Bypass Heat Recovery Steam Generator?',
       'RTO/ISO LMP Node Designation',
       'RTO/ISO Location Designation for Reporting Wholesale Sales Data to FERC',
       'Nameplate Capacity (MW)', 'Nameplate Power Factor',
       'Summer Capacity (MW)', 'Winter Capacity (MW)', 'Minimum Load (MW)',
       'Uprate or Derate Completed During Year',
       'Month Uprate or Derate Completed', 'Year Uprate or Derate Completed',
       'Status', 'Synchronized to Transmission Grid', 'Operating Month',
       'Operating Year', 'Planned Retirement Month', 'Planned Retirement Year',
       'Associated with Combined Heat and Power System', 'Sector Name',
       'Sector', 'Topping or Bottoming', 'Energy Source 1', 'Energy Source 2',
       'Energy Source 3', 'Energy Source 4', 'Energy Source 5',
       'Ene

In [19]:
renewables = generators[
    generators["Energy Source 1"].isin(["WND","SUN","WAT"])
]

In [20]:
renewables = renewables.merge(
    plants_clean,
    on="Plant Code",
    how="left"
)

In [28]:
renewables_clean = renewables[[
    "Plant Code",
    "Plant Name_y",
    "State_y",
    "Latitude",
    "Longitude",
    "Energy Source 1",
    "Nameplate Capacity (MW)"
]]

In [29]:
renewables_clean = renewables_clean.drop_duplicates(subset="Plant Code")

In [31]:
renewables_clean = renewables_clean.rename(columns={
    "Plant Name_y": "Plant Name",
    "State_y": "State",
    "Energy Source 1": "Energy Source"
})

In [34]:
renewables_clean["Plant Code"] = renewables_clean["Plant Code"].astype(int)

In [35]:
renewables_clean.head()

,Plant Code,Plant Name,State,Latitude,Longitude,Energy Source,Nameplate Capacity (MW)
0,1,Sand Point,AK,55.339722,-160.497222,WND,0.5
2,2,Bankhead Dam,AL,33.458665,-87.356823,WAT,53.9
3,4,Walter Bouldin Dam,AL,32.583889,-86.283056,WAT,75.0
6,11,H Neely Henry Dam,AL,33.7845,-86.0524,WAT,24.3
9,12,Holt Dam,AL,33.2553,-87.4495,WAT,46.9


In [36]:
renewables_clean["Energy Source"].value_counts()

Energy Source
SUN    6492
WAT    1457
WND    1346
Name: count, dtype: int64

In [39]:
renewables_clean.isna().sum()

Plant Code                 0
Plant Name                 1
State                      1
Latitude                   1
Longitude                  1
Energy Source              0
Nameplate Capacity (MW)    0
dtype: int64

In [40]:
renewables_clean = renewables_clean.dropna(subset=["Latitude", "Longitude"])

In [41]:
renewables_clean.isna().sum()

Plant Code                 0
Plant Name                 0
State                      0
Latitude                   0
Longitude                  0
Energy Source              0
Nameplate Capacity (MW)    0
dtype: int64

In [42]:
renewables_clean.duplicated(subset="Plant Code").sum()

np.int64(0)